# 06 — Minimal repro: vLLM `draft_model` cannot load compressed-tensors checkpoints

**Purpose**: a small, self-contained, publicly-runnable reproduction of the bug
found in `docs/findings.md` 2026-07-24 (`ValueError: ... weight_packed ...`) while
trying to use our own SGT-QAT Qwen3-1.7B checkpoint as a vLLM speculative-decoding
drafter for Qwen3-8B. That original repro used a private checkpoint (Google Drive)
and a mixed-precision (W4/W3) recipe.

**Update, 2026-07-26 — the plain-checkpoint simplification was wrong.** Section 3
(plain, single-scheme W4A16, no mixed precision) loaded successfully in vLLM — no
error. The bug is **not** "compressed-tensors checkpoints in general." Section 5
tests the actual remaining hypothesis: that it's specifically **mixed-precision**
(`config_groups` with different bit-widths per layer subset) compressed-tensors
checkpoints that fail. Do not assume section 5 will reproduce it either — run it
and see. `docs/vllm-bug-report-draft.md` must not be filed until one of these
cells actually reproduces the failure and the report's claim matches whichever
one it turns out to be.</cell id="cell-0">


## 1. Environment info for the bug report

Run this first — `docs/vllm-bug-report-draft.md` has placeholders for both outputs
below. Paste them back in exactly as printed, don't summarize/truncate.

In [ ]:
import os
!pip install -q vllm

# Known issue (docs/logs.md 2026-07-23, hit in notebooks 02/03): pip can resolve
# a CUDA-13-linked vLLM binary alongside a CUDA-12.x torch build -- `import vllm`
# then fails with `ImportError: libcudart.so.13`. Fix it here, before anything
# imports torch/vllm, in case this fresh environment hits the same mismatch.
import glob, subprocess
cu13_libs = glob.glob('/usr/local/lib/python3.*/dist-packages/nvidia/cu13/lib/libcudart.so.13')
if cu13_libs and not os.path.exists('/usr/lib/x86_64-linux-gnu/libcudart.so.13'):
    subprocess.run(['ln', '-sf', cu13_libs[0], '/usr/lib/x86_64-linux-gnu/libcudart.so.13'], check=True)
    subprocess.run(['ldconfig'], check=True)
    print(f"Symlinked {cu13_libs[0]} -> /usr/lib/x86_64-linux-gnu/libcudart.so.13 (CUDA runtime mismatch fix)")

In [ ]:
!pip show vllm
print("\n" + "="*80 + "\n")
!python -m vllm.collect_env

## 2. Build a tiny compressed-tensors checkpoint (plain W4A16 GPTQ, no mixed precision)

In [ ]:
!pip install -q llmcompressor compressed-tensors

import torch
from pathlib import Path
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

MODEL_ID = 'Qwen/Qwen3-0.6B'  # small on purpose -- this repro doesn't need our 1.7B/8B pair
SEQ_LEN = 2048
CALIB_N = 32  # small on purpose -- this is a loading-path bug, not a quality benchmark
CHECKPOINT_DIR = Path('checkpoints/qwen3-0.6b-w4a16-compressed-repro')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='cuda', trust_remote_code=True)

ds = load_dataset('allenai/c4', 'en', split='train', streaming=True).shuffle(seed=42, buffer_size=10_000)
samples, collected = [], 0
for item in ds:
    enc = tokenizer(item['text'], return_tensors='pt', truncation=True, max_length=SEQ_LEN)
    if enc['input_ids'].shape[1] == SEQ_LEN:
        samples.append(enc['input_ids'])
        collected += 1
        if collected >= CALIB_N:
            break
calib_dataset = Dataset.from_dict({'input_ids': torch.cat(samples, dim=0).tolist()})

# Plain W4A16, single scheme, no config_groups/mixed precision -- the simplest
# possible thing that still produces a genuinely packed compressed-tensors checkpoint.
recipe = GPTQModifier(targets='Linear', ignore=['lm_head'], scheme='W4A16', dampening_frac=0.01)
oneshot(model=model, dataset=calib_dataset, recipe=recipe, max_seq_length=SEQ_LEN, num_calibration_samples=CALIB_N)

model.save_pretrained(str(CHECKPOINT_DIR), save_compressed=True)
tokenizer.save_pretrained(str(CHECKPOINT_DIR))

# Sanity check: confirm this genuinely saved packed weights, not a full-precision
# fallback -- look for weight_packed in the safetensors keys before even trying
# to load it into vLLM.
from safetensors import safe_open
shard = next(CHECKPOINT_DIR.glob('*.safetensors'))
with safe_open(str(shard), framework='pt') as f:
    keys = list(f.keys())
has_packed = any('weight_packed' in k for k in keys)
print(f"Checkpoint saved to {CHECKPOINT_DIR}, contains weight_packed tensors: {has_packed}")
assert has_packed, "Checkpoint didn't actually save as compressed -- nothing to repro here."

del model
torch.cuda.empty_cache()

## 3. Trigger the bug: load it as a vLLM `draft_model`

Expected (per the original bug): this raises `ValueError: There is no module or
parameter named '...weight_packed' in ... . The available parameters belonging
to ... are: {'...weight'}`. **Actual result (2026-07-26): this did NOT raise —
the plain single-scheme checkpoint loaded successfully.** See the updated intro
cell and section 5 for the mixed-precision hypothesis this points to instead.</cell id="cell-6">


In [ ]:
import os
os.environ['VLLM_ENABLE_V1_MULTIPROCESSING'] = '0'  # so the real traceback surfaces here, not in a swallowed subprocess
os.environ['VLLM_LOGGING_LEVEL'] = 'DEBUG'

from vllm import LLM

llm = LLM(
    model=MODEL_ID,  # same tiny model as target, for simplicity -- the bug is in draft-model loading, not target/draft compatibility
    speculative_config={
        'method': 'draft_model',
        'model': str(CHECKPOINT_DIR.resolve()),
        'num_speculative_tokens': 3,
    },
    max_model_len=2048,
)

## 4. Copy the exact traceback from the cell above into `docs/vllm-bug-report-draft.md`

Paste the full traceback (not just the last line) into the "Actual behavior"
section, and the outputs from cell 1 into "Your current environment".

## 5. If section 3 did NOT raise an error: test the mixed-precision hypothesis

**Real result, 2026-07-26**: the plain single-scheme W4A16 checkpoint above
loaded successfully — no `ValueError`, full engine init completed. The minimal
repro did **not** reproduce the original bug.

This means the plain-W4A16 simplification likely removed the actual trigger.
Our real checkpoint used a **mixed-precision** recipe (`config_groups` with
*different* bit-widths for different layer subsets — W4 on protected layers,
W3 on the rest — see `notebooks/01`), not a single uniform scheme. That's a
structurally different compressed-tensors checkpoint (per-layer scheme
metadata, not one global scheme), and may be what vLLM's `draft_model` path
actually can't handle — not "compressed-tensors in general."

This cell tests that directly: same tiny model, same cheap calibration, but
with two `config_groups` (mirroring notebook 01's actual structure) instead of
one uniform scheme.

**Update, 2026-07-27 — split by full decoder-layer boundary, not flat list
order.** The first version of this cell split `all_linear_names` 50/50 by flat
list order. Qwen3 fuses `gate_proj`+`up_proj` into one merged weight
internally (`MergedColumnParallelLinear`); a flat 50/50 split can land
*inside* a layer, putting `gate_proj` and `up_proj` of the same layer in
different bit-width groups — a real confound, unrelated to the actual
question. The version below splits by whole decoder-layer index instead
(layers 0..mid-1 entirely W4, mid..end entirely W3), so no merged weight ever
straddles two bit-widths. This was still not enough to get a clean load — see
section 6a's 2026-07-27 update for what actually blocked it next.</cell id="a77cd874">


In [ ]:
import torch
from pathlib import Path
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

MIXED_CHECKPOINT_DIR = Path('checkpoints/qwen3-0.6b-w4w3-mixed-compressed-repro')
MIXED_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model_mixed = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='cuda', trust_remote_code=True)

# Reuse the same calibration approach as section 2 -- small on purpose.
ds = load_dataset('allenai/c4', 'en', split='train', streaming=True).shuffle(seed=42, buffer_size=10_000)
samples, collected = [], 0
for item in ds:
    enc = tokenizer(item['text'], return_tensors='pt', truncation=True, max_length=SEQ_LEN)
    if enc['input_ids'].shape[1] == SEQ_LEN:
        samples.append(enc['input_ids'])
        collected += 1
        if collected >= CALIB_N:
            break
calib_dataset = Dataset.from_dict({'input_ids': torch.cat(samples, dim=0).tolist()})

# Split by whole decoder-layer boundary, not flat list order (2026-07-27 fix --
# see the markdown above this cell). Guarantees no merged weight (e.g.
# gate_up_proj) ever straddles two bit-widths within the same layer.
import re

all_linear_names = [name for name, m in model_mixed.named_modules()
                     if isinstance(m, torch.nn.Linear) and name != 'lm_head']

def layer_idx(name):
    m = re.match(r'model\.layers\.(\d+)\.', name)
    return int(m.group(1)) if m else -1

num_layers = max(layer_idx(n) for n in all_linear_names) + 1
mid_layer = num_layers // 2
group_w4 = [n for n in all_linear_names if layer_idx(n) < mid_layer]
group_w3 = [n for n in all_linear_names if layer_idx(n) >= mid_layer]
print(f"{len(group_w4)} layers -> W4 (layers 0-{mid_layer-1}), {len(group_w3)} layers -> W3 (layers {mid_layer}-{num_layers-1})")

mixed_recipe = GPTQModifier(
    dampening_frac=0.01, ignore=['lm_head'],
    config_groups={
        'w4_group': {
            'targets': group_w4,
            'input_activations': None, 'output_activations': None,
            'weights': {'num_bits': 4, 'type': 'int', 'symmetric': True, 'strategy': 'group', 'group_size': 128},
        },
        'w3_group': {
            'targets': group_w3,
            'input_activations': None, 'output_activations': None,
            'weights': {'num_bits': 3, 'type': 'int', 'symmetric': True, 'strategy': 'group', 'group_size': 128},
        },
    },
)
oneshot(model=model_mixed, dataset=calib_dataset, recipe=mixed_recipe, max_seq_length=SEQ_LEN, num_calibration_samples=CALIB_N)

model_mixed.save_pretrained(str(MIXED_CHECKPOINT_DIR), save_compressed=True)
tokenizer.save_pretrained(str(MIXED_CHECKPOINT_DIR))

from safetensors import safe_open
shard = next(MIXED_CHECKPOINT_DIR.glob('*.safetensors'))
with safe_open(str(shard), framework='pt') as f:
    keys = list(f.keys())
has_packed = any('weight_packed' in k for k in keys)
print(f"Mixed-precision checkpoint saved to {MIXED_CHECKPOINT_DIR}, contains weight_packed tensors: {has_packed}")
assert has_packed, "Checkpoint didn't actually save as compressed -- nothing to repro here."

del model_mixed
torch.cuda.empty_cache()

### 5b. Trigger: load the mixed-precision checkpoint as a `draft_model`

**Restart the runtime before running this section** (Runtime -> Restart
session in Colab, then re-run the install cell + section 5's two cells, skip
section 3). vLLM's engine holds persistent CUDA context/state that may not
clean up cleanly between two separate `LLM()` instantiations in one process —
a fresh process avoids a false pass/fail caused by leftover state from
section 3's already-successful load, rather than the actual bug.

If this raises the `weight_packed` `ValueError` where section 3's plain
checkpoint didn't, that confirms the bug is specifically about mixed-precision
(`config_groups`) compressed-tensors checkpoints, not compressed-tensors
checkpoints in general — and `docs/vllm-bug-report-draft.md` needs rewriting
around that narrower, more precise claim before filing.

In [ ]:
import os
os.environ['VLLM_ENABLE_V1_MULTIPROCESSING'] = '0'
os.environ['VLLM_LOGGING_LEVEL'] = 'DEBUG'

from vllm import LLM

llm_mixed = LLM(
    model=MODEL_ID,
    speculative_config={
        'method': 'draft_model',
        'model': str(MIXED_CHECKPOINT_DIR.resolve()),
        'num_speculative_tokens': 3,
    },
    max_model_len=2048,
)

## 6. Verify the maintainer's fix (PR #49900)

Filed as [vllm-project/vllm#49893](https://github.com/vllm-project/vllm/issues/49893).
Maintainer `harjothkhara` root-caused it fast and matches our own finding:
the draft model loads under a `draft_model` prefix at runtime, which breaks
exact-name/anchored-regex `config_groups` target matching — single-scheme
worked because `Linear`-class-name matching is substring-based, mixed-precision
(name-based) targets weren't. Fix PR:
[#49900](https://github.com/vllm-project/vllm/pull/49900), Python-only (no
compile needed).

**2026-07-27 update — status after actually running this section:**
- **Point 1: CONFIRMED.** The mixed-precision checkpoint no longer hits the
  original `weight_packed` `ValueError` at all — the fix works for the issue
  as filed.
- **New blocker found, reported back on the PR.** Loading now gets much
  further (past layer 14) but hits a different `AssertionError` in
  `vllm/model_executor/parameter.py:175` (`load_merged_column_weight`) —
  `param_data.shape` (`[3072, 96]`) doesn't match `loaded_weight.shape`
  (`[3072, 103]`) when loading a merged column-parallel weight (looks like
  `gate_up_proj`). Confirmed via `%debug` post-mortem, not a guess. This
  persisted even after ruling out our own test-construction as the cause
  (section 5's layer-boundary-aligned split, so no merged weight straddles
  two bit-widths). See section 6a's update below for the full diagnostic and
  the comment posted on the PR. **Points 2-4 are still blocked** behind this
  — can't get to generation/memory checks until the mixed-precision load
  itself completes cleanly.
- **Environment note (real cost, not a repro issue)**: `llmcompressor`
  requires `torch<=2.12.0`; this fix branch's source build requires
  `torch==2.13.0` exactly. These cannot coexist in one environment. Building
  the checkpoints (needs `llmcompressor`) and loading them into the fixed
  `vllm` (needs the newer torch) must happen in **two separate Colab
  sessions**, handed off via Google Drive — see the new subsection below
  before section 6a. Installing `llmcompressor` into a session that already
  has the fix branch's `vllm` installed will silently downgrade torch and
  break `vllm`/`torchvision` — don't do it, even to "just rebuild one
  checkpoint quickly."

In [ ]:
# Attempt 1: precompiled install, per the maintainer's exact command. As of
# 2026-07-27, this 404s -- the nightly wheel index has nothing published for
# the commit setup.py auto-resolves, in any CUDA variant. Reported on the PR.
# Left here for reference / to retry if their wheel infra catches up, but
# expect it to fail and fall through to Attempt 2 below.
#
# !pip uninstall -y vllm -q
# !VLLM_USE_PRECOMPILED=1 pip install -q "git+https://github.com/harjothkhara/vllm.git@oss-find/vllm-2026-07-26"

# Attempt 2 (what actually worked, 2026-07-27): full source build, no
# VLLM_USE_PRECOMPILED. Takes ~1hr on a Colab A100 -- real compute cost, but
# bypasses their nightly-wheel infra entirely and builds the actual PR code.
!pip uninstall -y vllm -q
!pip install "git+https://github.com/harjothkhara/vllm.git@oss-find/vllm-2026-07-26"

!pip show vllm | grep -E "^(Name|Version)"

# After this cell: RESTART THE RUNTIME (pip will tell you to). The build also
# pulls in a newer torch as a dependency, which breaks torchaudio's import-time
# CUDA-version check -- fix below, run once after restarting.

In [ ]:
# Run once, after restarting the runtime post-install. torchaudio wasn't
# rebuilt against the newer torch pulled in above, so `from vllm import LLM`
# (which touches torchaudio's extension loader indirectly) fails with a
# CUDA-version mismatch RuntimeError. Not needed for text generation here.
!pip uninstall -y torchaudio -q

**If you ever accidentally run `pip install llmcompressor` in *this* session**
(don't — see the Drive-handoff section below): it silently downgrades torch
to satisfy its own `<=2.12.0` pin, which breaks `torchvision` (still linked
against 2.13.0) and can leave `pyarrow` in a broken state (mismatched
compiled `.so` vs. package metadata, `ValueError: pyarrow.lib.IpcReadOptions
size changed...`). If you hit that: `!pip uninstall -y pyarrow -q && !rm -rf
/usr/local/lib/python3.12/dist-packages/pyarrow* && !pip install
--no-cache-dir pyarrow` (clean purge, not just `--force-reinstall`, which
doesn't fully overwrite it), then `!pip install --no-cache-dir torch==2.13.0`
to restore what `vllm`/`torchvision` need. Better to just not install
`llmcompressor` here at all — that's what the next section is for.

### 6-prep. Get the checkpoints here without installing llmcompressor in this session

If sections 2/5 already ran earlier **in this same Colab session, before**
installing the fix branch's `vllm`, the checkpoints are already on local disk
— skip to 6a directly. Otherwise (fresh session, or checkpoints got lost to a
VM/session recycle — Colab will silently reconnect to a new backend after a
long idle/build period, wiping `/content` but not necessarily changing what
`pip show vllm` reports if you reinstalled after reconnecting):

1. In a **separate** Colab session (new tab, fresh runtime, `vllm` never
   installed there) run sections 2 and 5's build cells only — cheap, a few
   minutes each on the tiny model.
2. In that session, copy both checkpoints to Drive:
   ```python
   from google.colab import drive
   drive.mount('/content/drive')
   !mkdir -p /content/drive/MyDrive/vllm-bug-repro-checkpoints
   !cp -r checkpoints/qwen3-0.6b-w4a16-compressed-repro /content/drive/MyDrive/vllm-bug-repro-checkpoints/
   !cp -r checkpoints/qwen3-0.6b-w4w3-mixed-compressed-repro /content/drive/MyDrive/vllm-bug-repro-checkpoints/
   !du -sh checkpoints/qwen3-0.6b-w4a16-compressed-repro /content/drive/MyDrive/vllm-bug-repro-checkpoints/qwen3-0.6b-w4a16-compressed-repro
   !du -sh checkpoints/qwen3-0.6b-w4w3-mixed-compressed-repro /content/drive/MyDrive/vllm-bug-repro-checkpoints/qwen3-0.6b-w4w3-mixed-compressed-repro
   ```
   **Verify the `du -sh` pairs match** before moving on — Drive copies of
   large files have been flaky mid-copy elsewhere in this project too (see
   `docs/context.md` "Checkpoint storage").
3. Back in **this** (vllm) session, mount Drive and copy them in locally:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# mkdir -p first, so BOTH cp calls land as proper subdirectories -- cp's
# behavior differs depending on whether the destination already exists
# (learned the hard way, 2026-07-27): if checkpoints/ doesn't exist yet, the
# first `cp -r src checkpoints/` creates checkpoints/ AS a copy of src's
# contents (flattened), not nested inside it.
!rm -rf checkpoints
!mkdir -p checkpoints
!cp -r /content/drive/MyDrive/vllm-bug-repro-checkpoints/qwen3-0.6b-w4a16-compressed-repro checkpoints/
!cp -r /content/drive/MyDrive/vllm-bug-repro-checkpoints/qwen3-0.6b-w4w3-mixed-compressed-repro checkpoints/
!find checkpoints -maxdepth 2
!du -sh checkpoints/qwen3-0.6b-w4a16-compressed-repro checkpoints/qwen3-0.6b-w4w3-mixed-compressed-repro

# Confirm model.safetensors shows up in both before proceeding to 6a.

In [ ]:
# Redefine these after any runtime restart -- restarting wipes Python
# variables but not local disk, so this is cheap and just needs re-running.
from pathlib import Path
MODEL_ID = 'Qwen/Qwen3-0.6B'
CHECKPOINT_DIR = Path('checkpoints/qwen3-0.6b-w4a16-compressed-repro')
MIXED_CHECKPOINT_DIR = Path('checkpoints/qwen3-0.6b-w4w3-mixed-compressed-repro')

### 6a. Point 1 + setup for point 4: load the mixed-precision (compressed) checkpoint

Reuses `MIXED_CHECKPOINT_DIR` from the Drive-restore step above. Records GPU
memory before/after for the point-4 comparison later in this section.

**2026-07-27 result**: gets past the original `weight_packed` `ValueError`
entirely (point 1 confirmed) — but crashes further in with a NEW
`AssertionError` in `vllm/model_executor/parameter.py:175`
(`load_merged_column_weight`):
```
assert param_data.shape == loaded_weight.shape
AssertionError:
```
Diagnosed via `%debug` post-mortem (run `%debug` in a fresh cell immediately
after this one fails, don't re-run first):
```
p param_data.shape    -> torch.Size([3072, 96])
p loaded_weight.shape -> torch.Size([3072, 103])
p self.output_dim     -> 0
p shard_size          -> 3072
p self.tp_rank        -> 0
```
Single GPU (`tp_rank=0`), sharded dim (3072) matches on both sides — the
mismatch is entirely in the packed/quantized dimension. Ruled out our own
test construction as the cause first (this is the layer-boundary-aligned
checkpoint from the updated section 5, not the original flat-split one, and
it fails identically). Looks like `param_data` (vLLM's pre-allocated buffer
for this merged weight, likely `gate_up_proj`) assumes one packing width for
the whole fused parameter, but the checkpoint's `gate_proj`/`up_proj`
sub-tensors may carry different quantization metadata once there's more than
one `config_groups` scheme in the checkpoint. Reported on the PR with these
exact numbers — not something to keep debugging further on our end for now.
**Points 2-4 below are blocked until this is resolved upstream.**</cell id="be419a2d">


In [ ]:
import os, subprocess
os.environ['VLLM_ENABLE_V1_MULTIPROCESSING'] = '0'
os.environ['VLLM_LOGGING_LEVEL'] = 'DEBUG'

def _gpu_memory_used_bytes(device_index: int = 0) -> int:
    out = subprocess.check_output([
        'nvidia-smi', f'--id={device_index}',
        '--query-gpu=memory.used', '--format=csv,noheader,nounits',
    ])
    return int(out.decode().strip().splitlines()[0]) * 1024 * 1024

baseline_mem_compressed = _gpu_memory_used_bytes()

from vllm import LLM

llm_mixed_fixed = LLM(
    model=MODEL_ID,
    speculative_config={
        'method': 'draft_model',
        'model': str(MIXED_CHECKPOINT_DIR.resolve()),
        'num_speculative_tokens': 3,
    },
    max_model_len=2048,
)

loaded_mem_compressed = _gpu_memory_used_bytes()
delta_compressed = loaded_mem_compressed - baseline_mem_compressed
print(f"POINT 1: mixed-precision checkpoint loaded successfully: {llm_mixed_fixed is not None}")
print(f"In-serving memory delta (compressed mixed-precision drafter): {delta_compressed / 1024**3:.3f} GiB")

### 6b. Point 3: a short generation actually runs on the mixed-precision drafter

In [ ]:
from vllm import SamplingParams

outputs = llm_mixed_fixed.generate(
    ["The capital of France is"],
    SamplingParams(max_tokens=32, temperature=0),
)
for out in outputs:
    print("PROMPT:", out.prompt)
    print("OUTPUT:", out.outputs[0].text)

print(f"\nPOINT 3: generation completed without error: {len(outputs) > 0}")

### 6c. Point 4 (second half): compare against the decompressed workaround

Decompresses the same mixed-precision checkpoint the way `notebooks/03` had
to (before this fix existed), loads *that* as the drafter instead, and
compares its in-serving memory delta against section 6a's compressed-checkpoint
number. If the fix genuinely preserves compression through loading, the
compressed delta from 6a should be meaningfully smaller than this one.

**Restart the runtime before this cell** (same persistent-CUDA-context/two-`LLM()`
caution as before) — re-run the fix-install cell first, skip 6a/6b, land here.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from compressed_tensors import ModelCompressor
import torch.nn as nn

PLAIN_MIXED_CHECKPOINT_DIR = Path('checkpoints/qwen3-0.6b-w4w3-mixed-plain-repro')

def _unwrap_quantized_linears(model):
    # same approach as notebooks/03 -- see docs/findings.md 2026-07-24
    replaced = 0
    for name, module in list(model.named_modules()):
        if hasattr(module, 'weight_scale') or hasattr(module, 'weight_shape'):
            parent_name, _, child_name = name.rpartition('.')
            parent = model.get_submodule(parent_name) if parent_name else model
            weight = module.weight.data
            has_bias = getattr(module, 'bias', None) is not None
            plain = nn.Linear(weight.shape[1], weight.shape[0], bias=has_bias, dtype=weight.dtype)
            plain.weight.data.copy_(weight)
            if has_bias:
                plain.bias.data.copy_(module.bias.data)
            setattr(parent, child_name, plain)
            replaced += 1
    print(f"Replaced {replaced} quantized modules with plain nn.Linear.")

if not PLAIN_MIXED_CHECKPOINT_DIR.exists():
    _decompress_model = AutoModelForCausalLM.from_pretrained(
        str(MIXED_CHECKPOINT_DIR), dtype=torch.float16, trust_remote_code=True
    )
    _compressor = ModelCompressor.from_pretrained_model(_decompress_model)
    _compressor.decompress_model(_decompress_model)
    _unwrap_quantized_linears(_decompress_model)
    if getattr(_decompress_model, 'hf_quantizer', None) is not None:
        _decompress_model.hf_quantizer.remove_quantization_config(_decompress_model)
    if hasattr(_decompress_model.config, 'quantization_config'):
        _decompress_model.config.quantization_config = None

    _decompress_tokenizer = AutoTokenizer.from_pretrained(str(MIXED_CHECKPOINT_DIR))
    PLAIN_MIXED_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    _decompress_model.save_pretrained(str(PLAIN_MIXED_CHECKPOINT_DIR), save_original_format=False)
    _decompress_tokenizer.save_pretrained(str(PLAIN_MIXED_CHECKPOINT_DIR))
    del _decompress_model
    torch.cuda.empty_cache()

import os, subprocess
os.environ['VLLM_ENABLE_V1_MULTIPROCESSING'] = '0'

def _gpu_memory_used_bytes(device_index: int = 0) -> int:
    out = subprocess.check_output([
        'nvidia-smi', f'--id={device_index}',
        '--query-gpu=memory.used', '--format=csv,noheader,nounits',
    ])
    return int(out.decode().strip().splitlines()[0]) * 1024 * 1024

baseline_mem_plain = _gpu_memory_used_bytes()

from vllm import LLM

llm_plain_workaround = LLM(
    model=MODEL_ID,
    speculative_config={
        'method': 'draft_model',
        'model': str(PLAIN_MIXED_CHECKPOINT_DIR.resolve()),
        'num_speculative_tokens': 3,
    },
    max_model_len=2048,
)

loaded_mem_plain = _gpu_memory_used_bytes()
delta_plain = loaded_mem_plain - baseline_mem_plain
print(f"In-serving memory delta (decompressed/plain workaround drafter): {delta_plain / 1024**3:.3f} GiB")
print(f"\nPOINT 4: compare this against section 6a's compressed-checkpoint delta")
print(f"(re-run/reference that number here manually -- separate runtime session).")

### 6d. Point 2: confirm no regression on the single-scheme checkpoint

**Restart the runtime before this cell.** Re-run the fix-install cell first,
then this one — reuses `CHECKPOINT_DIR` from section 2 (already on disk).

In [ ]:
import os
os.environ['VLLM_ENABLE_V1_MULTIPROCESSING'] = '0'

from vllm import LLM

llm_single_scheme_fixed = LLM(
    model=MODEL_ID,
    speculative_config={
        'method': 'draft_model',
        'model': str(CHECKPOINT_DIR.resolve()),
        'num_speculative_tokens': 3,
    },
    max_model_len=2048,
)

print(f"POINT 2: single-scheme checkpoint still loads (no regression): {llm_single_scheme_fixed is not None}")

## 7. Report back on the issue

**2026-07-27: done for now.** Posted the point-1-confirmed / new-AssertionError
finding (exact shapes from the `%debug` session, see 6a above) as a comment on
[vllm-project/vllm#49893](https://github.com/vllm-project/vllm/issues/49893) —
full text also kept in `docs/vllm-bug-report-draft.md`. Points 2-4 (single-scheme
regression check, generation, memory comparison) are still blocked behind this
new finding and haven't been run. Once the maintainer responds (either a fix for
the merged-weight shape issue, or clarification that this is a separate/known
limitation), pick back up at 6a with whatever's needed, then continue to 6b-6d.
Real numbers only when reporting anything further — same rule as everywhere
else in this project.</cell id="403fea10">
